# RetailPulse -- Data Drift Detection

**Objective:** Detect data drift between training and recent data using PSI and KS tests.

In [1]:
import os, warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")
FIGURES_DIR = os.path.join("..", "reports", "figures")
PROCESSED_DIR = os.path.join("..", "data", "processed")
def save_fig(fig, name):
    fig.savefig(os.path.join(FIGURES_DIR, name), dpi=150, bbox_inches="tight", facecolor="white")
    plt.close(fig); print(f"Saved: {name}")


In [2]:
daily = pd.read_csv(os.path.join(PROCESSED_DIR, "daily_sales_features.csv"), parse_dates=["Date"])
split = len(daily) - 90
reference = daily.iloc[:split]
current = daily.iloc[split:]
print(f"Reference period: {reference['Date'].min().date()} to {reference['Date'].max().date()} ({len(reference)} days)")
print(f"Current period: {current['Date'].min().date()} to {current['Date'].max().date()} ({len(current)} days)")


Reference period: 2009-12-01 to 2011-09-10 (649 days)
Current period: 2011-09-11 to 2011-12-09 (90 days)


## Population Stability Index (PSI)

PSI measures distribution shift. PSI < 0.1 = no drift, 0.1-0.25 = moderate, > 0.25 = significant.

In [3]:
def calculate_psi(reference, current, bins=10):
    ref_hist, bin_edges = np.histogram(reference, bins=bins)
    cur_hist, _ = np.histogram(current, bins=bin_edges)
    ref_pct = (ref_hist + 1) / (ref_hist.sum() + bins)
    cur_pct = (cur_hist + 1) / (cur_hist.sum() + bins)
    psi = np.sum((cur_pct - ref_pct) * np.log(cur_pct / ref_pct))
    return psi

drift_features = ["total_revenue", "total_quantity", "transaction_count", "unique_customers", "avg_order_value"]
drift_results = []
for feat in drift_features:
    psi = calculate_psi(reference[feat].dropna(), current[feat].dropna())
    ks_stat, ks_pval = stats.ks_2samp(reference[feat].dropna(), current[feat].dropna())
    status = "No Drift" if psi < 0.1 else ("Moderate" if psi < 0.25 else "Significant Drift")
    drift_results.append({"Feature": feat, "PSI": round(psi, 4), "KS Statistic": round(ks_stat, 4),
                          "KS p-value": round(ks_pval, 4), "Status": status})

drift_df = pd.DataFrame(drift_results)
print("DATA DRIFT ANALYSIS")
print("=" * 70)
print(drift_df.to_string(index=False))


DATA DRIFT ANALYSIS
          Feature    PSI  KS Statistic  KS p-value            Status
    total_revenue 1.0695        0.4561      0.0000 Significant Drift
   total_quantity 0.7897        0.4549      0.0000 Significant Drift
transaction_count 1.1600        0.4695      0.0000 Significant Drift
 unique_customers 1.1773        0.5016      0.0000 Significant Drift
  avg_order_value 0.1985        0.2033      0.0024          Moderate


In [4]:
fig, axes = plt.subplots(2, 3, figsize=(20, 10))
axes = axes.flatten()
for i, feat in enumerate(drift_features):
    axes[i].hist(reference[feat].dropna(), bins=20, alpha=0.6, color="#3498db", label="Reference", density=True)
    axes[i].hist(current[feat].dropna(), bins=20, alpha=0.6, color="#e74c3c", label="Current", density=True)
    psi_val = drift_df[drift_df["Feature"] == feat]["PSI"].values[0]
    axes[i].set_title(f"{feat}\nPSI={psi_val:.4f}"); axes[i].legend(fontsize=8)
axes[-1].axis("off")
fig.suptitle("Distribution Comparison: Reference vs Current", fontsize=16, fontweight="bold", y=1.01)
fig.tight_layout(); save_fig(fig, "38_drift_analysis.png"); plt.show()


Saved: 38_drift_analysis.png


In [5]:
drift_df.to_csv(os.path.join(PROCESSED_DIR, "drift_report.csv"), index=False)
print("Saved: drift_report.csv")
print("\nDRIFT DETECTION COMPLETE")


Saved: drift_report.csv

DRIFT DETECTION COMPLETE
